<a href="https://colab.research.google.com/github/menna890/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0.Setup


In [ ]:
import os
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"


def build_active_from_warehouse(con, rel, month="2026-03"):
    daily = con.sql(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position
        FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/**/*.parquet')
        WHERE gsc_data_available IS TRUE
    """).df()

    pages = (
        daily.groupby(["client_hash_id", "content_hash_id"], as_index=False)
        .agg(
            impressions=("gsc_impressions", "sum"),
            clicks=("gsc_clicks", "sum"),
            avg_position=("gsc_avg_position", "mean"),
        )
    )
    pages["ctr"] = np.where(
        pages["impressions"] > 0,
        100.0 * pages["clicks"] / pages["impressions"],
        np.nan,
    )
    active = pages[
        (pages["impressions"] >= 100)
        & (pages["avg_position"].notna())
        & (pages["avg_position"] > 0)
    ].copy()

    def position_band(p):
        if p <= 3:
            return "1-3"
        if p <= 10:
            return "4-10"
        if p <= 20:
            return "11-20"
        return "21+"

    active["position_band"] = active["avg_position"].map(position_band)
    active["ctr_band_median"] = active.groupby("position_band")["ctr"].transform("median")
    active["gap_ratio"] = np.where(
        active["ctr_band_median"] > 0,
        (active["ctr_band_median"] - active["ctr"]) / active["ctr_band_median"],
        0.0,
    )

    # Same baseline rule as ML-07
    focus = active["position_band"].isin(["1-3", "4-10"])
    positive_gap = active["gap_ratio"] > 0
    active["baseline_score"] = 0.0
    active.loc[focus & positive_gap, "baseline_score"] = (
        active.loc[focus & positive_gap, "gap_ratio"]
        * np.log1p(active.loc[focus & positive_gap, "impressions"])
    )

    # Label used for supervised ranking evaluation
    active["is_gap"] = (
        active["position_band"].isin(["1-3", "4-10"])
        & (active["gap_ratio"] >= 0.30)
    ).astype(int)

    return active


active = build_active_from_warehouse(con, rel)
print(active.shape)
print("is_gap rate:", round(active["is_gap"].mean(), 4))
print("baseline_score > 0:", int((active["baseline_score"] > 0).sum()))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(101441, 11)
is_gap rate: 0.2272
baseline_score > 0: 27942


## 1. Method choice and why

Decision: rank pages for first human review of a page-1 CTR gap
(CTR below the position-band median, with enough impressions).

Methods:
- Logistic Regression — readable reference
- Random Forest — stronger tabular interactions

Training target: is_gap (observed proxy from GSC aggregates on the March 2026 slice).
Deployment-style use: rank by P(is_gap=1) and evaluate Precision@K.

### Honest framing (important)

Our label `is_gap` is derived from the same CTR-band idea as the Week-4 rule
(gap_ratio ≥ 0.30 within focus bands). The rule’s `baseline_score` ranks with
gap_ratio and impressions. Therefore, comparing that rule’s ranking against
learned models *on the same label* is partly circular: the rule is close to the
label definition by construction.

What we report instead, with clear interpretation:

1. Precision@K for learned models (Logistic, RF) vs `is_gap`
2. Comparison to naive references (random scores; test base rate)
3. The rule’s Precision@K is shown for transparency, but is **not** treated as a
   fair “model beat the rule” contest on this label

Research question for the learned models:
*With only limited features (avg_position, impressions, position_band)—no CTR—
can a model still produce a useful decision-support ranking of gap pages?*

We keep complexity only if it clearly beats naive ranking, not because it is more complex.

### 2. Split design

GroupShuffleSplit by client_hash_id (test_size=0.25, random_state=42).

Honest for this lane because pages share client-level templates and CTR regimes.
IDs are groups only — never model features.
March 2026 development slice only; June sealed month not used for labels.

In [ ]:
FEATURE_NUM = ["avg_position", "impressions"]
FEATURE_CAT = ["position_band"]
# Optional later: word_count_filled, has_word_count — not required to start honest

y = active["is_gap"].astype(int)
groups = active["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(active, y, groups))
train = active.iloc[train_idx].copy()
test = active.iloc[test_idx].copy()

assert len(set(train["client_hash_id"]) & set(test["client_hash_id"])) == 0

print(f"Train {len(train):,} | Test {len(test):,}")
print(f"Train clients {train['client_hash_id'].nunique()} | Test clients {test['client_hash_id'].nunique()}")
print(f"Base rate train {train['is_gap'].mean():.4f} | test {test['is_gap'].mean():.4f}")

Train 93,085 | Test 8,356
Train clients 33 | Test clients 11
Base rate train 0.2321 | test 0.1730


## 3. Train + compare (honest reading)

Same test rows for all methods. Metric: Precision@20, @50, @100 + test base rate.

| Method | Role in this notebook |
|---|---|
| Random scores | Naive reference |
| Baseline rule (baseline_score) | Shown for transparency; near-circular with is_gap — not a fair “beat the rule” target |
| Logistic / Random Forest | Main learned rankers (features: position, impressions, band only — no CTR) |

Primary finding we care about: do RF/Logit beat random / base rate with limited features?
Secondary: rule P@K is expected to be very high on this label; we do not claim the model must beat it.

In [ ]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(int(k), len(scores))
    if k == 0:
        return np.nan
    top = np.argsort(-scores)[:k]
    return float(y_true[top].mean())


pre = ColumnTransformer(
    [
        ("num", "passthrough", FEATURE_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURE_CAT),
    ]
)

X_train = train[FEATURE_NUM + FEATURE_CAT]
X_test = test[FEATURE_NUM + FEATURE_CAT]
y_train = train["is_gap"].astype(int).to_numpy()
y_test = test["is_gap"].astype(int).to_numpy()

logit = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

rf = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

logit.fit(X_train, y_train)
rf.fit(X_train, y_train)

p_logit = logit.predict_proba(X_test)[:, 1]
p_rf = rf.predict_proba(X_test)[:, 1]
s_base = test["baseline_score"].to_numpy()

rng = np.random.RandomState(RANDOM_STATE)
s_random = rng.rand(len(y_test))
base_rate = float(y_test.mean())

rows = []
for name, scores in [
    ("Random scores (noise ref)", s_random),
    ("Baseline rule (baseline_score)", s_base),
    ("Logistic Regression", p_logit),
    ("Random Forest", p_rf),
]:
    rows.append({
        "method": name,
        "precision@20": precision_at_k(y_test, scores, 20),
        "precision@50": precision_at_k(y_test, scores, 50),
        "precision@100": precision_at_k(y_test, scores, 100),
    })

compare = pd.DataFrame(rows)
compare["test_base_rate"] = base_rate
print(compare.round(4).to_string(index=False))

os.makedirs("work/outputs", exist_ok=True)
compare.to_csv("work/outputs/ml08_model_vs_baseline.csv", index=False)

# Ranked model queue on test (decision output)
test_out = test[
    ["client_hash_id", "content_hash_id", "avg_position", "impressions", "ctr",
     "gap_ratio", "baseline_score", "is_gap"]
].copy()
test_out["p_logit"] = p_logit
test_out["p_rf"] = p_rf
test_out = test_out.sort_values("p_rf", ascending=False).reset_index(drop=True)
test_out["rank_rf"] = np.arange(1, len(test_out) + 1)
test_out.head(50).to_csv("work/outputs/ml08_test_top50_rf.csv", index=False)
print("Wrote work/outputs/ml08_model_vs_baseline.csv and ml08_test_top50_rf.csv")

                        method  precision@20  precision@50  precision@100  test_base_rate
     Random scores (noise ref)          0.10          0.12           0.16           0.173
Baseline rule (baseline_score)          1.00          1.00           1.00           0.173
           Logistic Regression          0.60          0.54           0.51           0.173
                 Random Forest          0.85          0.70           0.63           0.173
Wrote work/outputs/ml08_model_vs_baseline.csv and ml08_test_top50_rf.csv


## 4. Errors and interpretation

RF top-50: false positives and hard misses.

False positives often have *good* CTR (negative gap_ratio) but page-1-like rank/volume —
the model cannot see CTR, so it confuses “looks like a gap page” with “is a gap page”.

Hard misses include strong top-3 gaps with solid volume — the rule would rank these high;
the no-CTR model under-scores some of them.

Feature importances lean on avg_position and band (especially 4–10), then impressions.
No label columns in X (leak check empty).

### Finding (decision-support, observational)

- Random Forest Precision@20 ≈ 0.85 vs test base rate ≈ 0.17 and random ≈ 0.10:
  rank + volume + band carry real signal for prioritising gap-like pages *without CTR*.
- The CTR-aware rule reaches Precision@K ≈ 1.0 on this label by construction; we do **not**
  claim the model should replace that rule when CTR is available at decision time.
- Operational recommendation: use the Week-4 rule queue when GSC CTR is available;
  treat the no-CTR model as evidence that partial ranking is still possible from rank/volume alone.

Claims: measured on a client-grouped March 2026 test split; directional decision-support only —
not a forecast of Google’s ranking algorithm or guaranteed traffic lift.

In [ ]:
eval_df = test.copy()
eval_df["p_rf"] = p_rf

top50 = eval_df.nlargest(50, "p_rf")
fp = top50[top50["is_gap"] == 0]
fn = eval_df[eval_df["is_gap"] == 1].nsmallest(15, "p_rf")

print("RF top50 precision:", round(top50["is_gap"].mean(), 4))
print("False positives in top50:", len(fp))
print("\nFP sample:")
print(fp[["content_hash_id", "avg_position", "impressions", "ctr", "gap_ratio", "p_rf"]].head(5))

print("\nHard misses (true gaps, low score):")
print(fn[["content_hash_id", "avg_position", "impressions", "ctr", "gap_ratio", "p_rf"]].head(5))

# Importances
ohe = rf.named_steps["pre"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(FEATURE_CAT))
feat_names = FEATURE_NUM + cat_names
imp = pd.DataFrame({
    "feature": feat_names,
    "importance": rf.named_steps["clf"].feature_importances_,
}).sort_values("importance", ascending=False)
print("\nFeature importance:")
print(imp.to_string(index=False))

# Sanity: features must not include label leakage columns
leak_cols = {"is_gap", "gap_ratio", "ctr", "baseline_score", "ctr_band_median"}
used = set(FEATURE_NUM + FEATURE_CAT)
print("Leak columns in features (must be empty):", sorted(used & leak_cols))

RF top50 precision: 0.7
False positives in top50: 15

FP sample:
                 content_hash_id  avg_position  impressions       ctr  \
148450  content_9da4bb4be41cd2f9      8.090705          108  0.925926   
37099   content_c25a3ddc5478a05b      8.086123          116  0.862069   
58770   content_410f2ad937e4623f      0.599376          317  1.892744   
131672  content_1a1b121694b32b63      7.938724          124  1.612903   
35221   content_5da0e2f14a96a9e2      7.181797          109  0.917431   

        gap_ratio      p_rf  
148450  -3.805556  0.939340  
37099   -3.474138  0.933562  
58770   -6.886435  0.931959  
131672  -7.370968  0.931447  
35221   -3.761468  0.930708  

Hard misses (true gaps, low score):
                content_hash_id  avg_position  impressions       ctr  \
60926  content_d0c1810bc7a536f9      3.388714         3358  0.059559   
60365  content_ab8f5fff84a1b794      3.188594         2883  0.069372   
58678  content_3b1aa1bff5199d75      3.206283         8982  0.1

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.